# 第 1 章:文本数据处理与 Tokenization

**核心问题**:神经网络只能处理数字,文本如何变成模型能吃的整数序列?

**章节内容**
1. 简单分词策略(字符级、词级)—— 感受它们的局限
2. BPE(字节对编码)原理 —— 用 ~30 行代码实现
3. 用 `tiktoken`(OpenAI 官方 BPE)对比验证
4. 封装 `Tokenizer` 类:对外暴露 `encode` / `decode`

**产出文件**:`ch01_tokenizer.ipynb`(本文件) + `llm/tokenizer.py`

In [1]:
# 全章通用导入
import re
from collections import Counter, defaultdict
from pprint import pprint

---
## 1. 简单分词策略

### 1.1 字符级分词

最直观:每个字符是一个 token。

- 优点:词表极小(ASCII 约 128 个),永远没有未登录词
- 缺点:序列很长,模型需要从字符学出「单词」的概念,难度大

In [3]:
text = "Hello, world! GPT is amazing."

# TODO: 字符级 encode —— 把 text 变成字符列表,再映射为整数 id
# 提示:先 sorted(set(text)) 建词表,再做 char->id 的字典

vocab = sorted(set(text))
char2id = {char: idx for idx, char in enumerate(vocab)}  # TODO: 字典推导式 {char: idx for idx, char in enumerate(vocab)}
id2char = {idx: char for char, idx in char2id.items()}  # TODO: 反向字典

encoded = [char2id[c] for c in text]  # TODO: [char2id[c] for c in text]
decoded = ''.join([id2char[i] for i in encoded])  # TODO: ''.join([id2char[i] for i in encoded])

print('词表大小:', len(vocab))
print('词表:', vocab)
print('编码前 10 个:', encoded[:10])
print('解码验证:', decoded == text)  # 应为 True

词表大小: 21
词表: [' ', '!', ',', '.', 'G', 'H', 'P', 'T', 'a', 'd', 'e', 'g', 'i', 'l', 'm', 'n', 'o', 'r', 's', 'w', 'z']
编码前 10 个: [5, 10, 13, 13, 16, 2, 0, 19, 16, 17]
解码验证: True


### 1.2 词级分词(空格切割)

- 优点:序列短,token 有完整语义
- 缺点:
  - **词表爆炸**:真实语料词表可达几十万
  - **未登录词(OOV)**:训练时没见过的词无法处理
  - 无法处理形态变化:`run` vs `running` 是两个不同 token

In [5]:
corpus = 'the cat sat on the mat\nthe cat sat on the hat\nthe dog sat on the log'

# TODO: 词级分词
# 1) 用 str.split() 切出所有词
# 2) 建词表 (sorted + set)
# 3) 建 word2id / id2word
# 4) 编码 corpus 的第一行

words = corpus.split()
vocab_words = sorted(set(words))
word2id = {char: idx for idx, char in enumerate(vocab_words)}
id2word = {idx: char for char, idx in word2id.items()}

first_line = corpus.split('\n')[0].split()
encoded_line = [word2id[c] for c in first_line]
print('词表:', vocab_words)
print('第一行编码:', encoded_line)

# 思考:如果新文本出现了 'frog',会怎样?试一下 word2id.get('frog', -1)
# 答：返回-1（哨兵值）

词表: ['cat', 'dog', 'hat', 'log', 'mat', 'on', 'sat', 'the']
第一行编码: [7, 0, 6, 5, 7, 4]


---
## 2. BPE(Byte-Pair Encoding)原理

BPE 是字符级和词级的折中:
- 从字符出发,把**最高频的相邻字节对**反复合并成新 token
- 高频词会被合并成完整 token(`the` -> 一个 id)
- 低频词/新词会保留为更小的子词(`unbelievable` -> `un` + `believ` + `able`)

### 算法流程

```
初始:把每个词切成字符序列,词尾加 </w> 标记
循环 N 次:
    1. 统计所有相邻字节对的出现频率
    2. 找最高频的对 (a, b)
    3. 把所有 (a, b) 合并为 ab
    4. 把合并规则 (a, b) -> ab 记入 merge_rules
结束后词表 = 初始字符集 + 所有 merge_rules 产生的新 token
```

In [ ]:
# ── 辅助函数(已给出,先理解再往下) ──────────────────────────────────

def get_vocab(corpus):
    """把语料切成 word_str -> freq 字典。每个词字符间加空格,词尾加 </w>。"""
    word_freqs = Counter(corpus.lower().split())
    vocab = {}
    for word, freq in word_freqs.items():
        token = ' '.join(list(word)) + ' </w>'
        vocab[token] = freq
    return vocab

def get_pairs(vocab):
    """统计所有相邻字节对的总频率。"""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    """把 vocab 中所有出现 pair 的地方合并为一个新 token。"""
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab

# ── 演示语料 ──────────────────────────────────────────────────────────
demo_corpus = 'low lower lowest newer newer newest'

vocab = get_vocab(demo_corpus)
print('初始词表:')
pprint(vocab)

In [ ]:
# TODO: 完成 BPE 训练循环
#
# 目标:合并 10 次,每次打印合并了哪对
#
# 伪代码:
#   for i in range(num_merges):
#       pairs = get_pairs(vocab)
#       if not pairs: break
#       best = ...           # pairs.most_common(1)[0][0]
#       vocab = merge_vocab(best, vocab)
#       merge_rules.append(best)
#       print(f'第 {i+1} 次合并: {best} -> {"" .join(best)}')

num_merges = 10
merge_rules = []

# 你的代码:


In [ ]:
# TODO: 用训练好的 merge_rules 对新词做 BPE encode
#
# 思路:
#   1) 把新词切成字符列表并加 </w>
#   2) 按 merge_rules 的顺序依次尝试合并相邻符号
#   3) 返回最终的 token 列表

def bpe_encode(word, merge_rules):
    symbols = list(word) + ['</w>']
    for (a, b) in merge_rules:
        new_symbols = []
        i = 0
        while i < len(symbols):
            # TODO: 如果 symbols[i]==a 且下一个符号==b,就合并成 a+b,否则保留
            pass
        symbols = new_symbols
    return symbols

for word in ['low', 'lowest', 'newer', 'unknown']:
    tokens = bpe_encode(word, merge_rules)
    print(f'{word:10s} -> {tokens}')

# 观察:low/lowest/newer 训练语料里频率高,会被合并成完整 token
#       unknown 是未登录词,会保留为细粒度子词

---
## 3. 用 `tiktoken` 对比

`tiktoken` 是 OpenAI 开源的 BPE 实现,GPT-2/GPT-3/GPT-4 都用它。
编码方案 `gpt2` 的词表大小是 **50,257**。

这一节用 tiktoken 验证 BPE 思路,同时熟悉实际 tokenizer 的行为。

In [ ]:
import tiktoken

enc = tiktoken.get_encoding('gpt2')
print('词表大小:', enc.n_vocab)  # 50257

sample = 'Hello, world! GPT is amazing.'

# TODO: 用 enc.encode(sample) 得到 token id 列表
ids = ...
print('Token ids:', ids)
print('Token 数量:', len(ids))

# TODO: 用 enc.decode(ids) 还原文本
decoded = ...
print('还原文本:', decoded)
print('是否相等:', decoded == sample)

In [ ]:
# 可视化:每个 token 对应哪段文本?
print('Token 拆解:')
for token_id in ids:
    token_bytes = enc.decode_single_token_bytes(token_id)
    try:
        token_str = token_bytes.decode('utf-8')
    except Exception:
        token_str = repr(token_bytes)
    print(f'  id={token_id:5d}  字节={token_bytes!r:20s}  文本={token_str!r}')

In [ ]:
# 实验 1:同一个词,加空格和不加空格 id 不同
print('不带空格:', enc.encode('hello'))
print('带空格:  ', enc.encode(' hello'))  # GPT-2 把空格也编进 token

# 实验 2:数字的编码
for n in ['1', '10', '100', '1000', '12345']:
    print(f'{n:8s} -> {enc.encode(n)}')

# 思考:为什么 100 可能是单个 token,而 12345 需要多个 token?

---
## 4. 封装 `Tokenizer` 类

后面所有章节都会用到 tokenizer。封装一个干净的接口:

```
tokenizer.encode('Hello world')   -> [15496, 995]
tokenizer.decode([15496, 995])     -> 'Hello world'
tokenizer.vocab_size               -> 50257
```

内部调用 `tiktoken`,但接口统一,以后可以替换实现。

In [ ]:
# TODO: 完成 Tokenizer 类

class Tokenizer:
    def __init__(self, encoding_name='gpt2'):
        import tiktoken
        self._enc = tiktoken.get_encoding(encoding_name)

    @property
    def vocab_size(self):
        # TODO: 返回词表大小
        ...

    def encode(self, text):
        # TODO: 返回 id 列表
        ...

    def decode(self, ids):
        # TODO: 返回字符串
        ...


# 验证
tokenizer = Tokenizer()
text = 'In the beginning God created the heavens and the earth.'
ids = tokenizer.encode(text)
print('编码:', ids)
print('解码:', tokenizer.decode(ids))
print('还原是否一致:', tokenizer.decode(ids) == text)
print('词表大小:', tokenizer.vocab_size)

In [ ]:
# 确认上面验证通过后运行这个 cell —— 把 Tokenizer 写入 llm/tokenizer.py

import pathlib

code = 'import tiktoken\n\n\nclass Tokenizer:\n    def __init__(self, encoding_name="gpt2"):\n        self._enc = tiktoken.get_encoding(encoding_name)\n\n    @property\n    def vocab_size(self):\n        return self._enc.n_vocab\n\n    def encode(self, text: str) -> list:\n        return self._enc.encode(text)\n\n    def decode(self, ids: list) -> str:\n        return self._enc.decode(ids)\n'

out = pathlib.Path('../llm/tokenizer.py')
out.write_text(code, encoding='utf-8')
print(f'已写入 {out.resolve()}')

# 验证可导入
import sys
sys.path.insert(0, str(out.parent.parent))
import importlib
import llm.tokenizer
importlib.reload(llm.tokenizer)
from llm.tokenizer import Tokenizer as T2
assert T2().decode(T2().encode('hello')) == 'hello'
print('llm.tokenizer 导入验证通过')

---
## 章末思考题

1. BPE 为什么能处理未登录词(OOV)?字符级和词级分别如何应对 OOV?
2. GPT-2 为什么把空格也编进 token(如 ` hello` 和 `hello` 是不同 id)?有什么好处?
3. 词表越大越好吗?词表大小对模型参数量有什么影响?(提示:Embedding 层大小 = vocab_size × embed_dim)

能答出来就可以进入第 2 章 —— DataLoader + 词嵌入 + 位置编码。